# Physically scaled nucleus classification

Only intrinsic single-nucleus morphology is used. Every row and prediction is one nucleus. Organoid IDs only prevent cross-validation leakage; nothing is aggregated. Results from the old mixed/unit-inconsistent tables are obsolete.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
DATA = ROOT/'data'/'recomputed_pure_nucleus_features'
OUT = ROOT/'results'/'classification'; OUT.mkdir(parents=True, exist_ok=True)
import matplotlib.pyplot as plt
import pandas as pd
from analysis.data import load_and_filter, get_feature_matrix
from analysis.features import PURE_NUCLEUS_FEATURES
from analysis.manual_vs_auto import load as load_domain, run_all as run_cross_domain
from run_classification_comparison import run_cv
PURE_NUCLEUS_FEATURES

## Input and scaling audit
Spacing/objective columns are retained for audit only and never enter the model.

In [ ]:
tables = {}
for trial in ('005','028'):
    path = DATA/f'trial_{trial}_p021n_p013t.csv'
    raw, filtered = pd.read_csv(path), load_and_filter(path)
    tables[trial] = filtered
    print(trial, 'raw=',len(raw),'filtered=',len(filtered),'images=',filtered.img_name.nunique(),'imputed=',int(raw.spacing_is_imputed.sum()))
    display(raw.groupby(['objective','spacing_z_um','spacing_y_um','spacing_x_um'],dropna=False).agg(n_nuclei=('label','size'),n_images=('img_name','nunique')).reset_index())

In [ ]:
fig, axes = plt.subplots(1,3,figsize=(14,4))
for ax, feature in zip(axes,['volume_um3','ellipsoid_axis_major_um','sphericity']):
    values=[tables[t][feature].clip(upper=tables[t][feature].quantile(.99)) for t in ('005','028')]
    ax.boxplot(values,tick_labels=['trial 005','trial 028'],showfliers=False); ax.set_title(feature)
fig.suptitle('Scaled nucleus features after quality filtering'); fig.tight_layout()

## Grouped nucleus-level cross-validation
`GroupKFold` keeps nuclei from one image together. Fast interpretable models run by default. Enable TabPFN if its weights are available; run TabFM and AutoGluon on suitable compute with `run_classification_comparison.py`.

In [ ]:
RUN_TABPFN=False
models=['logreg','l0l2_logreg']+(['tabpfn'] if RUN_TABPFN else [])
rows=[]
for trial,df in tables.items():
    X,y,groups=get_feature_matrix(df)
    for model in models:
        aucs,accs=run_cv(model,X,y,groups)['nucleus']
        rows.append({'trial':trial,'model':model,'auc_mean':aucs.mean(),'auc_std':aucs.std(),'accuracy_mean':accs.mean(),'accuracy_std':accs.std(),'n_folds':len(aucs)})
cv_results=pd.DataFrame(rows); cv_results.to_csv(OUT/'notebook_grouped_nucleus_cv.csv',index=False); cv_results

## Objective sensitivity
The 25× subset is P021N-only. Exclusion is therefore reported as a sensitivity analysis, never selected post hoc for better performance.

In [ ]:
rows=[]
for trial in ('005','028'):
    for exclude in (False,True):
        df=load_and_filter(DATA/f'trial_{trial}_p021n_p013t.csv',exclude_25x=exclude); X,y,g=get_feature_matrix(df)
        for model in ('logreg','l0l2_logreg'):
            aucs,_=run_cv(model,X,y,g)['nucleus']; rows.append({'trial':trial,'exclude_25x':exclude,'model':model,'auc_mean':aucs.mean(),'auc_std':aucs.std(),'n_nuclei':len(df)})
sensitivity=pd.DataFrame(rows); sensitivity.to_csv(OUT/'notebook_objective_sensitivity.csv',index=False); sensitivity

## Manual ↔ automated transfer
Automated rows are restricted to the exact 22 manually annotated images. Scoring remains at nucleus level.

In [ ]:
manual=load_domain(DATA/'manual_p021n_p013t.csv'); automated={}
for trial in ('005','028'):
    auto=load_domain(DATA/f'trial_{trial}_p021n_p013t.csv')
    automated[f'trial{trial}']=auto[auto.organoid_key.isin(manual.organoid_key.unique())].copy()
transfer=run_cross_domain(manual,automated,modes=('nucleus',)); transfer.to_csv(OUT/'notebook_manual_auto_nucleus_transfer.csv',index=False); transfer